In [1]:
import torch
import os
import glob
from unsloth import FastLanguageModel
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 512

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "deepseek-ai/deepseek-math-7b-rl"

tokenizer = AutoTokenizer.from_pretrained(
    "./deepseek_math_tokenizer"
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)



In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
print("Tokenizer vocab:", len(tokenizer))
print("Model vocab:", model.get_input_embeddings().weight.shape[0])
print("PAD ID:", tokenizer.pad_token_id)
print("EOS ID:", tokenizer.eos_token_id)

Tokenizer vocab: 100003
Model vocab: 102400
PAD ID: 100002
EOS ID: 100001


In [6]:

dataset = load_dataset("juletxara/mgsm", 'en', split = "train")
test_dataset = load_dataset("juletxara/mgsm", 'en', split = "test")

In [7]:
df = pd.DataFrame(dataset)
df[:21]


,question,answer,answer_number,equation_solution
0,Question: Roger has 5 tennis balls. He buys 2 ...,Step-by-Step Answer: Roger started with 5 ball...,11,5 + 6 = 11.
1,Question: There were nine computers in the ser...,Step-by-Step Answer: There are 4 days from mon...,29,4 * 5 = 20. 9 + 20 = 29.
2,Question: Leah had 32 chocolates and her siste...,Step-by-Step Answer: Leah had 32 chocolates an...,39,32 + 42 = 74. 74 - 35 = 39.
3,"Question: Shawn has five toys. For Christmas, ...",Step-by-Step Answer: He has 5 toys. He got 2 f...,9,5 + 2 = 7. 7 + 2 = 9.
4,Question: Michael had 58 golf balls. On tuesda...,Step-by-Step Answer: Michael started with 58 g...,33,58 - 23 = 35. 35 - 2 = 33.
5,Question: Olivia has $23. She bought five bage...,Step-by-Step Answer: 5 bagels for $3 each shou...,8,5 * 3 = 15. 23 - 15 = 8.
6,Question: Jason had 20 lollipops. He gave Denn...,Step-by-Step Answer: Jason started with 20 lol...,8,20 - 12 = 8.
7,Question: If there are 3 cars in the parking l...,Step-by-Step Answer: There are 3 cars in the b...,5,3 + 2 = 5.


In [ ]:
# from trl import SFTTrainer
# from transformers import TrainingArguments, DataCollatorForSeq2Seq
# from unsloth import is_bfloat16_supported



# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2, # Number of processors to use for processing the dataset
#     packing = False, # Can make training 5x faster for short sequences.
#     args = TrainingArguments(
#         per_device_train_batch_size = 8, # The batch size per GPU/TPU core
#         gradient_accumulation_steps = 8, # Number of steps to perform before each gradient accumulation
#         # warmup_steps = 5, # Few updates with low learning rate before actual training
#         # max_steps = 60, # Specifies the total number of training steps (batches) to run.
#         num_train_epochs=2,
#         warmup_ratio=0.3,
#         learning_rate = 2e-4,
#         fp16 = not is_bfloat16_supported(),
#         bf16 = is_bfloat16_supported(),
#         logging_steps = 1,
#         optim = "adamw_8bit", # Optimizer
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
        
#         save_strategy="steps",
#         save_steps=100,
#         save_total_limit=5, 

#         seed = 3407,
#         output_dir = "outputs",
#         report_to = "none", # Use this for WandB etc for observability
#     ),
# )

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py


In [12]:
import transformers
import trl
import unsloth
import accelerate

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("unsloth:", unsloth.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.57.6
trl: 0.24.0
unsloth: 2026.7.1
accelerate: 1.14.0


In [ ]:
# from unsloth.chat_templates import get_chat_template
# sys_prompt = """
# <problem>
# {}
# </problem>
# """
# message = sys_prompt.format("একটি বন্ধুকে পরীক্ষার প্রস্তুতির জন্য উৎসাহ দিয়ে ৩ লাইনের একটি বার্তা বাংলায় লেখো।")
# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "llama-3.1",
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#     {"role": "user", "content": message},
# ]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True,
#                          temperature = 1.5, min_p = 0.1)
# response = tokenizer.batch_decode(outputs)

In [7]:
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(100003, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRM

In [8]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def generate_response(prompt, system_prompt=None):

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    # Create input IDs + attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=512,
        temperature=0.7,
        min_p=0.1,
        use_cache=True,
    )

    # Remove the input tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # Decode only the generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [9]:
while True:
    prompt = input("\nYou: ")

    if prompt.lower() in ["exit", "quit"]:
        break

    response = generate_response(prompt)

    print("\nAssistant:", response)


Assistant: Roger initially has 5 tennis balls. He buys 2 more cans of tennis balls, and each can has 3 tennis balls. So, the total number of tennis balls he gets from the cans is 2 * 3 = 6.

To find the total number of tennis balls Roger has now, we add the number of tennis balls he initially had to the number of tennis balls he got from the cans. That is 5 + 6 = 11.

So, Roger now has 11 tennis balls. The answer is $\boxed{11}$.


In [10]:
def create_prompt(question):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve the following math problem. "
                "Show your reasoning and give the final answer clearly.\n\n"
                f"Question:\n{question}"
            )
        }
    ]

    # Use model's chat template if available
    if tokenizer.chat_template is not None:

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    else:

        prompt = (
            "Solve the following math problem. "
            "Show your reasoning and give the final answer clearly.\n\n"
            f"Question:\n{question}\n\n"
            "Answer:"
        )

    return prompt

In [11]:
question = test_dataset[0]["question"]

prompt = create_prompt(question)

print(prompt)

<｜begin▁of▁sentence｜>User: Solve the following math problem. Show your reasoning and give the final answer clearly.

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Assistant:


In [19]:
def collate_fn(batch):

    questions = [
        item["question"]
        for item in batch
    ]

    answers = [
        item["answer_number"]
        for item in batch
    ]

    prompts = [
        create_prompt(q)
        for q in questions
    ]

    return {
        "questions": questions,
        "answers": answers,
        "prompts": prompts
    }



dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [20]:
import re

def extract_numerical_answer(response):
    # Remove <think>...</think>
    response = re.sub(
        r"<think>.*?</think>",
        "",
        response,
        flags=re.DOTALL | re.IGNORECASE
    )

    # 1. Look for \boxed{...}
    matches = re.findall(
        r"\\boxed\{\s*(-?\d+(?:\.\d+)?)\s*\}",
        response
    )
    if matches:
        return float(matches[-1])

    # 2. Look for GSM8K format: #### 10
    matches = re.findall(
        r"####\s*(-?\d+(?:\.\d+)?)",
        response
    )
    if matches:
        return float(matches[-1])

    # 3. Look for "Final Answer"
    matches = re.findall(
        r"(?:Final Answer|final answer).*?"
        r"(-?\d+(?:\.\d+)?)",
        response,
        flags=re.DOTALL
    )
    if matches:
        return float(matches[-1])

    # 4. Fallback: last number in the response
    matches = re.findall(
        r"(?<![\w.])-?\d+(?:\.\d+)?(?![\w.])",
        response
    )
    if matches:
        return float(matches[-1])

    return None

In [21]:

CHECKPOINT_EVERY = 10
CHECKPOINT_DIR = "MGSM_en_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# Find latest checkpoint
# ============================================================

checkpoint_files = glob.glob(
    os.path.join(
        CHECKPOINT_DIR,
        "checkpoint_batch_*.csv"
    )
)


if checkpoint_files:

    def get_batch_number(path):
        filename = os.path.basename(path)
        match = re.search(
            r"checkpoint_batch_(\d+)\.csv",
            filename
        )
        return int(match.group(1))

    latest_checkpoint = max(
        checkpoint_files,
        key=get_batch_number
    )

    last_batch = get_batch_number(
        latest_checkpoint
    )

    print("Latest checkpoint found:")
    print(latest_checkpoint)
    print("Last completed batch:", last_batch)

else:

    latest_checkpoint = None
    last_batch = 0

    print("No checkpoint found.")
    print("Starting evaluation from the beginning.")

No checkpoint found.
Starting evaluation from the beginning.


In [22]:
if latest_checkpoint is not None:

    checkpoint_df = pd.read_csv(
        latest_checkpoint
    )

    results = checkpoint_df.to_dict(
        orient="records"
    )

    total = len(results)

    correct = sum(
        1 for result in results
        if result["correct"]
    )

    print("Checkpoint loaded.")
    print("Already processed:", total)
    print("Already correct:", correct)

    if total > 0:
        print(
            f"Current accuracy: "
            f"{correct / total * 100:.2f}%"
        )

else:

    results = []
    correct = 0
    total = 0

In [23]:
# Number of examples already processed
start_index = total

print("Starting from example:", start_index)
print("Remaining examples:", len(test_dataset) - start_index)


# Create a subset containing only unprocessed examples
remaining_dataset = test_dataset.select(
    range(start_index, len(test_dataset))
)


dataloader = DataLoader(
    remaining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


print("Remaining batches:", len(dataloader))

Starting from example: 0
Remaining examples: 250
Remaining batches: 63


In [24]:
for batch_idx, batch in enumerate(dataloader):

    questions = batch["questions"]
    reference_answers = batch["answers"]
    prompts = batch["prompts"]

    # --------------------------------------------------------
    # Tokenize batch
    # --------------------------------------------------------

    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    )

    # Move to GPU
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # Remove input tokens
    # --------------------------------------------------------

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[:, input_length:]

    predictions = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # --------------------------------------------------------
    # Evaluate examples
    # --------------------------------------------------------

    for question, reference_answer, prediction in zip(
        questions,
        reference_answers,
        predictions
    ):

        predicted_answer = extract_numerical_answer(
            prediction
        )


        is_correct = (
            predicted_answer == reference_answer
        )

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "model_output": prediction,
            "reference_answer": reference_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    current_accuracy = correct / total

    # Overall batch number
    current_batch = last_batch + batch_idx + 1

    print(
        f"Batch {current_batch} | "
        f"Examples: {total}/{len(test_dataset)} | "
        f"Accuracy: {current_accuracy * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Save checkpoint every 10 batches
    # --------------------------------------------------------

    if (batch_idx + 1) % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(results)

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"checkpoint_batch_{current_batch}.csv"
        )

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )

        print()
        print("=" * 60)
        print("CHECKPOINT SAVED")
        print("=" * 60)
        print("File:", checkpoint_path)
        print("Processed:", total)
        print(
            f"Accuracy: "
            f"{current_accuracy * 100:.2f}%"
        )
        print("=" * 60)
        print()

Batch 1 | Examples: 4/250 | Accuracy: 100.00%
Batch 2 | Examples: 8/250 | Accuracy: 87.50%
Batch 3 | Examples: 12/250 | Accuracy: 83.33%
Batch 4 | Examples: 16/250 | Accuracy: 81.25%
Batch 5 | Examples: 20/250 | Accuracy: 75.00%
Batch 6 | Examples: 24/250 | Accuracy: 70.83%
Batch 7 | Examples: 28/250 | Accuracy: 71.43%
Batch 8 | Examples: 32/250 | Accuracy: 71.88%
Batch 9 | Examples: 36/250 | Accuracy: 72.22%
Batch 10 | Examples: 40/250 | Accuracy: 75.00%

CHECKPOINT SAVED
File: MGSM_en_checkpoints/checkpoint_batch_10.csv
Processed: 40
Accuracy: 75.00%

Batch 11 | Examples: 44/250 | Accuracy: 77.27%
Batch 12 | Examples: 48/250 | Accuracy: 77.08%
Batch 13 | Examples: 52/250 | Accuracy: 76.92%
Batch 14 | Examples: 56/250 | Accuracy: 76.79%
Batch 15 | Examples: 60/250 | Accuracy: 78.33%
Batch 16 | Examples: 64/250 | Accuracy: 78.12%
Batch 17 | Examples: 68/250 | Accuracy: 79.41%
Batch 18 | Examples: 72/250 | Accuracy: 80.56%
Batch 19 | Examples: 76/250 | Accuracy: 78.95%
Batch 20 | Exampl

In [25]:
accuracy = correct / total

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(f"Correct : {correct}")
print(f"Total   : {total}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

FINAL RESULTS
Correct : 213
Total   : 250
Accuracy: 0.8520
Accuracy: 85.20%
